# Football Analytics Platform — Análisis de Arquero

Este notebook demuestra el flujo completo desde datos de tracking hasta insights tácticos.

**Audiencia:** Analistas tácticos, preparadores de arqueros, científicos de datos deportivos.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Módulos del sistema
from src.db.database import init_db, execute_raw
from src.analytics.metrics import calcular_metricas_jugador, generar_reporte_arquero
from src.features.feature_extractor import FeatureExtractor
from src.visualization.heatmaps import (
    heatmap_posicional, trazar_trayectoria,
    mapa_achiques, grafico_perfil_fisico, grafico_velocidad_tiempo
)

init_db(create_tables=True)
print('Sistema inicializado correctamente.')

## 1. Generar datos sintéticos para demostración

En producción, estos datos vienen del pipeline de videoanálisis (YOLO + ByteTrack).
Aquí simulamos el comportamiento realista de un arquero durante 90 minutos.

In [ ]:
np.random.seed(42)

FPS = 25.0
DURACION_MINUTOS = 90
TOTAL_FRAMES = int(FPS * DURACION_MINUTOS * 60)

# Simular posición de arquero: mayormente en área propia con salidas ocasionales
t = np.arange(TOTAL_FRAMES) / FPS  # segundos

# Posición base del arquero: x ~ 5-8m (área chica), y ~ 34m (centro)
x_base = 5.5 + np.random.normal(0, 1.5, TOTAL_FRAMES).cumsum() * 0.01
x_base = np.clip(x_base, 0.5, 16.5)  # dentro del área grande

y_base = 34.0 + np.random.normal(0, 0.5, TOTAL_FRAMES).cumsum() * 0.02
y_base = np.clip(y_base, 20.0, 48.0)  # dentro del área

# Agregar achiques (salidas fuera del área)
achiques_frames = np.random.choice(TOTAL_FRAMES, size=15, replace=False)
for f in achiques_frames:
    duracion = int(FPS * np.random.uniform(2, 6))  # 2-6 segundos
    fin = min(f + duracion, TOTAL_FRAMES)
    progreso = np.linspace(0, 1, fin - f)
    x_base[f:fin] = x_base[f] + progreso * np.random.uniform(10, 25)
    x_base[f:fin] = np.clip(x_base[f:fin], 0, 105)

# Velocidades
dx = np.diff(x_base, prepend=x_base[0])
dy = np.diff(y_base, prepend=y_base[0])
dist_frame = np.sqrt(dx**2 + dy**2)
velocidad_ms = dist_frame * FPS
velocidad_kmh = velocidad_ms * 3.6
aceleracion = np.diff(velocidad_ms, prepend=velocidad_ms[0]) * FPS

df_tracking = pd.DataFrame({
    'jugador_id': 1,
    'sesion_id': 1,
    'frame_numero': np.arange(TOTAL_FRAMES),
    'timestamp_seg': t,
    'x_campo': x_base,
    'y_campo': y_base,
    'velocidad_ms': np.clip(velocidad_ms, 0, 12),
    'velocidad_kmh': np.clip(velocidad_kmh, 0, 45),
    'aceleracion': aceleracion,
    'distancia_arco_propio': x_base,
    'presion_rival': np.random.beta(1.5, 5, TOTAL_FRAMES),  # distribución sesgada a baja presión
    'zona_id': None,
    'interpolado': False,
})

print(f'DataFrame generado: {len(df_tracking):,} filas × {len(df_tracking.columns)} columnas')
print(f'Período: {DURACION_MINUTOS} minutos a {FPS} fps')
df_tracking.describe().round(2)

## 2. Métricas Físicas

In [ ]:
metricas = calcular_metricas_jugador(
    df=df_tracking,
    jugador_id=1,
    sesion_id=1,
    fps=FPS,
    partido_id=1
)

print(generar_reporte_arquero(
    df_tracking=df_tracking,
    jugador_id=1,
    fps=FPS,
    nombre_arquero='Emiliano Martínez'
))

## 3. Heatmap Posicional

In [ ]:
fig = heatmap_posicional(
    df=df_tracking,
    titulo='Heatmap Posicional — Emiliano Martínez vs Rival FC',
    resolucion=2.5
)
plt.show()

## 4. Trayectoria Coloreada por Velocidad

In [ ]:
# Mostrar solo los primeros 30 minutos para claridad
df_30min = df_tracking[df_tracking['timestamp_seg'] <= 1800].copy()

fig = trazar_trayectoria(
    df=df_30min,
    titulo='Trayectoria — Primer Tiempo (coloreada por velocidad)',
    colorear_por='velocidad_kmh'
)
plt.show()

## 5. Mapa de Achiques

In [ ]:
fig = mapa_achiques(
    df=df_tracking,
    titulo='Mapa de Achiques — Zonas de salida del arquero'
)
plt.show()

## 6. Perfil Físico (Radar Chart)

In [ ]:
fig = grafico_perfil_fisico(
    metricas={
        'velocidad_max_kmh': metricas.velocidad_max_kmh,
        'sprints_total': metricas.sprints_total,
        'distancia_total_m': metricas.distancia_total_m,
        'achiques_realizados': metricas.achiques_realizados,
        'cobertura_lateral_m': metricas.cobertura_lateral_m,
        'aceleraciones_alta_int': metricas.aceleraciones_alta_int,
    },
    nombre='Emiliano Martínez'
)
plt.show()

## 7. Perfil de Velocidad por Tiempo

In [ ]:
fig = grafico_velocidad_tiempo(
    df=df_tracking,
    fps=FPS,
    titulo='Perfil de Velocidad — Partido Completo (90 min)'
)
plt.show()

## 8. Extracción de Features para ML

In [ ]:
extractor = FeatureExtractor(fps=FPS)
df_features = extractor.extraer(df_tracking)

print(f'Features generadas: {df_features.shape}')
print('\nColumnas disponibles:')
print(', '.join(df_features.columns.tolist()))

df_features[['frame_numero', 'velocidad_ms', 'aceleracion', 'jerk',
             'cambio_direccion_deg', 'distancia_arco_propio',
             'angulo_arco_rival', 'presion_rival']].head(10).round(4)

## 9. Análisis de Posicionamiento — Análisis Espacial

In [ ]:
# Distribución de posición X (profundidad)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de profundidad
axes[0].hist(df_tracking['x_campo'], bins=50, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[0].axvline(5.5,  color='yellow', linestyle='--', linewidth=2, label='Área chica (5.5m)')
axes[0].axvline(16.5, color='orange', linestyle='--', linewidth=2, label='Área grande (16.5m)')
axes[0].set_xlabel('Posición X (metros desde el arco propio)', fontsize=11)
axes[0].set_ylabel('Frecuencia (frames)', fontsize=11)
axes[0].set_title('Distribución de Profundidad del Arquero', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot comparativo por período
df_tracking['periodo'] = pd.cut(
    df_tracking['timestamp_seg'],
    bins=[0, 2700, 5400],
    labels=['Primer Tiempo', 'Segundo Tiempo']
)
df_tracking.boxplot(
    column='x_campo', by='periodo', ax=axes[1],
    boxprops=dict(color='#2ecc71'),
    whiskerprops=dict(color='#2ecc71'),
    medianprops=dict(color='red', linewidth=2),
)
axes[1].set_title('Posición por Período', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('X (metros)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 10. Correlación entre Features — Análisis para ML

In [ ]:
import seaborn as sns

features_correlacion = [
    'velocidad_ms', 'aceleracion', 'jerk',
    'distancia_arco_propio', 'distancia_arco_rival',
    'angulo_arco_rival', 'presion_rival',
    'velocidad_ms_media_5s'
]
features_disponibles = [f for f in features_correlacion if f in df_features.columns]

corr_matrix = df_features[features_disponibles].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Correlación entre Features — Arquero', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\nInsight: Alta correlación entre distancia_arco_propio y distancia_arco_rival')
print('(son complementarios para un campo de 105m) — considerar usar solo uno como feature.')